In [ ]:
import os, glob, gc, time, numpy as np, pandas as pd, torch, timm
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from tqdm import tqdm

LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
MODEL, EPOCHS, SEEDS = "densenet121", 7, [0, 1, 2]
NIH, OUT = r"G:\NIH", r"G:\runs"
ND = rf"{OUT}\nih_eval"; os.makedirs(ND, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{ND}\{name}.pdf", bbox_inches="tight")
    fig.savefig(rf"{ND}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

print("GPU:", torch.cuda.get_device_name(0))
print("models:", [os.path.exists(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}.pt") for s in SEEDS])

In [ ]:
paths = {}
for r, _, fs in os.walk(NIH):
    for n in fs:
        if n.lower().endswith(".png"):
            paths[n] = os.path.join(r, n)
print("PNG files found:", len(paths))

csv = glob.glob(rf"{NIH}\Data_Entry*.csv")[0]
d = pd.read_csv(csv).rename(columns={"Image Index": "name", "Finding Labels": "finding",
        "Patient ID": "pid", "Patient Age": "Age", "Patient Gender": "Sex",
        "View Position": "view", "Follow-up #": "followup"})
d = d[["name", "finding", "pid", "Age", "Sex", "view", "followup"]].copy()
d["patient_id"] = "nih" + d["pid"].astype(str)
d["Age"] = pd.to_numeric(d["Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
d.loc[d["Age"] > 100, "Age"] = np.nan
d["Sex"] = d["Sex"].map({"M": "Male", "F": "Female"})

MAP = {"Cardiomegaly": "Cardiomegaly", "Edema": "Edema", "Consolidation": "Consolidation",
       "Atelectasis": "Atelectasis", "Pneumothorax": "Pneumothorax", "Effusion": "Pleural Effusion"}
for k, v in MAP.items():
    d[v] = d["finding"].apply(lambda s, k=k: int(k in s.split("|")))

d["img_path"] = d["name"].map(paths)
print("rows without an image file:", d["img_path"].isna().sum())
d = d.dropna(subset=["img_path"]).reset_index(drop=True)

test_names = set(open(rf"{NIH}\test_list.txt").read().split())
d["official"] = np.where(d["name"].isin(test_names), "test", "train_val")
d.to_csv(r"G:\nih_clean.csv", index=False)
nih = d
print(len(nih), "images,", nih.patient_id.nunique(), "patients")
print(nih.official.value_counts().to_dict())
print(nih[LABELS].mean().round(3))

In [ ]:
cx = pd.read_csv(r"G:\chexpert_final_clean.csv")
cxt = cx[cx.split == "test"].reset_index(drop=True)

prev = pd.DataFrame({"CheXpert": cxt[LABELS].mean(), "NIH": nih[LABELS].mean()})
fig, ax = plt.subplots(figsize=(COL1 * 1.4, 2.6)); x = np.arange(len(LABELS))
ax.bar(x - 0.2, prev.CheXpert, 0.4, label="CheXpert", color="0.35")
ax.bar(x + 0.2, prev.NIH, 0.4, label="NIH", color="0.75", edgecolor="k", lw=0.4)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right")
ax.set(ylabel="Prevalence"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_prevalence")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
bins = np.arange(0, 101, 5)
ax[0].hist(cxt.Age.dropna(), bins, density=True, alpha=0.6, label="CheXpert", color="0.3")
ax[0].hist(nih.Age.dropna(), bins, density=True, alpha=0.6, label="NIH", color="0.7")
ax[0].set(xlabel="Age (years)", ylabel="Density"); ax[0].legend(frameon=False)
sx = pd.DataFrame({"CheXpert": cxt.Sex.value_counts(normalize=True), "NIH": nih.Sex.value_counts(normalize=True)})
sx.T[["Male", "Female"]].plot.bar(ax=ax[1], color=["0.3", "0.75"], edgecolor="k", lw=0.4, rot=0)
ax[1].set(ylabel="Fraction"); ax[1].legend(frameon=False)
for a in ax: a.grid(True, alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_demographics")

feat = nih[LABELS].copy(); feat["Age"] = nih.Age; feat["Sex (F=1)"] = (nih.Sex == "Female").astype(int)
feat["View (AP=1)"] = (nih["view"] == "AP").astype(int)
C = feat.corr().values; names = feat.columns.tolist()
fig, ax = plt.subplots(figsize=(COL2 * 0.7, COL2 * 0.65), constrained_layout=True)
im = ax.imshow(np.ma.array(C, mask=np.triu(np.ones_like(C, dtype=bool), k=1)), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(i + 1):
        ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(C[i, j]) > 0.6 else "black")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).set_label("Pearson $r$")
save_fig(fig, "fig_nih_correlation")

In [ ]:
CACHE, N = r"G:\nih_224.u8", len(nih)
mm = np.memmap(CACHE, dtype=np.uint8, mode="w+", shape=(N, 224, 224))
bad = []
def load(i):
    try:
        with Image.open(nih.img_path.iloc[i]) as im:
            mm[i] = np.asarray(im.convert("L").resize((224, 224), Image.BILINEAR))
    except Exception:
        bad.append(i)
with ThreadPoolExecutor(12) as pool:
    list(tqdm(pool.map(load, range(N)), total=N, desc="caching NIH at 224 px"))
mm.flush()
print("unreadable images:", len(bad))
np.save(rf"{ND}\bad_idx.npy", np.array(bad))

In [ ]:
arr = np.memmap(CACHE, dtype=np.uint8, mode="r", shape=(len(nih), 224, 224))
def load224(p):
    with Image.open(p) as im:
        return np.asarray(im.convert("L").resize((224, 224), Image.BILINEAR))
rng = np.random.default_rng(0)
ni, ci = rng.choice(len(nih), 5, replace=False), rng.choice(len(cxt), 5, replace=False)
fig, ax = plt.subplots(2, 5, figsize=(COL2, 3.0))
for j in range(5):
    ax[0, j].imshow(load224(cxt.img_path[ci[j]]), cmap="gray"); ax[1, j].imshow(arr[ni[j]], cmap="gray")
    ax[0, j].axis("off"); ax[1, j].axis("off")
ax[0, 0].set_title("CheXpert (top) / NIH (bottom)", loc="left", fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_samples")

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device="cuda").view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device="cuda").view(1, 3, 1, 1)
def prep(a):
    x = torch.from_numpy(np.ascontiguousarray(a)).cuda().float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD

def load_model(seed):
    m = timm.create_model(MODEL, pretrained=False, num_classes=len(LABELS)).cuda()
    m.load_state_dict(torch.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{seed}.pt", map_location="cuda", weights_only=True))
    return m.eval()

@torch.no_grad()
def predict_cache(m, a, bs=256):
    out = []
    for i in range(0, len(a), bs):
        with torch.autocast("cuda"):
            o = m(prep(a[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

NP = {}
for s in SEEDS:
    f = rf"{ND}\seed{s}_nih_preds.npy"
    if os.path.exists(f): NP[s] = np.load(f); continue
    m = load_model(s); NP[s] = predict_cache(m, arr); np.save(f, NP[s])
    del m; gc.collect(); torch.cuda.empty_cache()
    print("seed", s, "done")

Yn = nih[LABELS].values.astype(int)
Zc = {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz") for s in SEEDS}
Yc = Zc[0]["test_Y"].astype(int); Pc = {s: Zc[s]["test_P"] for s in SEEDS}
assert (Yc == cxt[LABELS].values).all(), "CheXpert predictions do not line up with the test table!"

def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

off = (nih.official == "test").values
rows = []
for s in SEEDS:
    for i, l in enumerate(LABELS):
        rows.append({"seed": s, "Label": l, "CheXpert": fauc(Yc[:, i], Pc[s][:, i]),
                     "NIH (all)": fauc(Yn[:, i], NP[s][:, i]),
                     "NIH (official test)": fauc(Yn[off, i], NP[s][off, i])})
res = pd.DataFrame(rows)
pm = lambda x: f"{x.mean():.3f} ± {x.std():.3f}"
tab = res.groupby("Label", sort=False)[["CheXpert", "NIH (all)", "NIH (official test)"]].agg(pm).loc[LABELS]
tab.loc["Mean"] = res.groupby("seed")[["CheXpert", "NIH (all)", "NIH (official test)"]].mean().agg(pm)
display(tab)
tab.to_csv(rf"{ND}\table_cx_vs_nih.csv")

In [ ]:
B = 300
def groups(pid):
    return list(pd.Series(np.arange(len(pid))).groupby(np.asarray(pid)).apply(np.array).values)
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values)
rng = np.random.default_rng(0)
bc, bn = np.zeros((B, len(LABELS))), np.zeros((B, len(LABELS)))
for b in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    inn = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for i in range(len(LABELS)):
        bc[b, i] = np.mean([fauc(Yc[ic, i], Pc[s][ic, i]) for s in SEEDS])
        bn[b, i] = np.mean([fauc(Yn[inn, i], NP[s][inn, i]) for s in SEEDS])
cx_pt = np.array([np.mean([fauc(Yc[:, i], Pc[s][:, i]) for s in SEEDS]) for i in range(len(LABELS))])
nh_pt = np.array([np.mean([fauc(Yn[:, i], NP[s][:, i]) for s in SEEDS]) for i in range(len(LABELS))])
gap_b = bc - bn
rows = []
for i, l in enumerate(LABELS):
    lo, hi = np.percentile(gap_b[:, i], [2.5, 97.5])
    rows.append({"Label": l, "CheXpert": cx_pt[i], "NIH": nh_pt[i], "Gap": cx_pt[i] - nh_pt[i], "CI low": lo, "CI high": hi})
lo, hi = np.percentile(gap_b.mean(1), [2.5, 97.5])
rows.append({"Label": "Mean", "CheXpert": cx_pt.mean(), "NIH": nh_pt.mean(), "Gap": cx_pt.mean() - nh_pt.mean(), "CI low": lo, "CI high": hi})
gap = pd.DataFrame(rows); display(gap.round(4))
gap.to_csv(rf"{ND}\table_gap_bootstrap.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(COL1 * 1.5, 2.7)); x = np.arange(len(LABELS))
ci_c = np.percentile(bc, [2.5, 97.5], axis=0); ci_n = np.percentile(bn, [2.5, 97.5], axis=0)
ax.bar(x - 0.2, cx_pt, 0.4, yerr=[np.clip(cx_pt - ci_c[0], 0, None), np.clip(ci_c[1] - cx_pt, 0, None)], capsize=2, color="0.35", label="CheXpert (in-hospital)")
ax.bar(x + 0.2, nh_pt, 0.4, yerr=[np.clip(nh_pt - ci_n[0], 0, None), np.clip(ci_n[1] - nh_pt, 0, None)], capsize=2, color="0.75", edgecolor="k", lw=0.4, label="NIH (external)")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right")
ax.set(ylabel="AUROC", ylim=(0.5, 0.95)); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_auroc_cx_vs_nih")

g = gap.iloc[:-1]
fig, ax = plt.subplots(figsize=(COL1 * 1.3, 2.5))
ax.bar(x, g.Gap, yerr=[np.clip(g.Gap - g["CI low"], 0, None), np.clip(g["CI high"] - g.Gap, 0, None)], capsize=2, color="0.5", edgecolor="k", lw=0.4)
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right")
ax.set(ylabel="AUROC drop (CheXpert - NIH)"); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_gap")

fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.2))
for i, (a, l) in enumerate(zip(axes.ravel(), LABELS)):
    f1_, t1_, _ = roc_curve(Yc[:, i], Pc[0][:, i]); f2_, t2_, _ = roc_curve(Yn[:, i], NP[0][:, i])
    a.plot(f1_, t1_, "-", label=f"CheXpert ({cx_pt[i]:.3f})"); a.plot(f2_, t2_, "--", label=f"NIH ({nh_pt[i]:.3f})")
    a.plot([0, 1], [0, 1], "k:", lw=0.6); a.set_title(l); a.grid(True, alpha=0.3)
    a.set(xlabel="False positive rate", ylabel="True positive rate"); a.legend(loc="lower right", frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_roc_cx_vs_nih")

def reliab(y, p, nb=10):
    b = np.clip((p * nb).astype(int), 0, nb - 1); xs, ys, ws = [], [], []
    for k in range(nb):
        m = b == k
        if m.sum() > 20: xs.append(p[m].mean()); ys.append(y[m].mean()); ws.append(m.mean())
    return np.array(xs), np.array(ys), np.array(ws)
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.2)); ece = []
for i, (a, l) in enumerate(zip(axes.ravel(), LABELS)):
    row = {"Label": l}
    for nm, Y_, P_, st in [("CheXpert", Yc, Pc[0], "-o"), ("NIH", Yn, NP[0], "--s")]:
        xs, ys, ws = reliab(Y_[:, i], P_[:, i]); a.plot(xs, ys, st, ms=2.5, label=nm)
        row["ECE " + nm] = float((ws * np.abs(xs - ys)).sum())
    ece.append(row); a.plot([0, 1], [0, 1], "k:", lw=0.6); a.set_title(l); a.grid(True, alpha=0.3)
    a.set(xlabel="Predicted probability", ylabel="Observed frequency"); a.legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_calibration")
ece = pd.DataFrame(ece); display(ece.round(3)); ece.to_csv(rf"{ND}\table_ece.csv", index=False)

fig, axes = plt.subplots(2, 3, figsize=(COL2, 3.8))
for i, (a, l) in enumerate(zip(axes.ravel(), LABELS)):
    a.hist(Pc[0][:, i], 40, range=(0, 1), density=True, alpha=0.55, label="CheXpert", color="0.3")
    a.hist(NP[0][:, i], 40, range=(0, 1), density=True, alpha=0.55, label="NIH", color="0.7")
    a.set_title(l); a.set(xlabel="Predicted probability", ylabel="Density"); a.set_yscale("log"); a.legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_score_distribution")

thr = np.mean([np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\thresholds.npy") for s in SEEDS], axis=0)
rows = []
for i, l in enumerate(LABELS):
    for nm, Y_, P_ in [("CheXpert", Yc, Pc[0]), ("NIH", Yn, NP[0])]:
        pr = (P_[:, i] >= thr[i]).astype(int)
        rows.append({"Label": l, "Set": nm, "Prevalence": Y_[:, i].mean(), "Precision": precision_score(Y_[:, i], pr, zero_division=0),
                     "Recall": recall_score(Y_[:, i], pr, zero_division=0), "F1": f1_score(Y_[:, i], pr, zero_division=0)})
thr_tab = pd.DataFrame(rows); display(thr_tab.round(3)); thr_tab.to_csv(rf"{ND}\table_fixed_threshold.csv", index=False)

In [ ]:
Pm = np.mean([NP[s] for s in SEEDS], axis=0)
G = {"Male": nih.Sex == "Male", "Female": nih.Sex == "Female", "Age <18": nih.Age < 18,
     "Age 18-49": nih.Age.between(18, 49), "Age 50-65": nih.Age.between(50, 65), "Age >65": nih.Age > 65,
     "PA view": nih["view"] == "PA", "AP view": nih["view"] == "AP"}
rows = []
for gname, msk in G.items():
    m = msk.values; r = {"Group": gname, "N": int(m.sum())}
    for i, l in enumerate(LABELS):
        r[l] = fauc(Yn[m, i], Pm[m, i]) if 0 < Yn[m, i].sum() < m.sum() else np.nan
    r["Mean"] = np.nanmean([r[l] for l in LABELS]); rows.append(r)
sub = pd.DataFrame(rows); display(sub.round(3)); sub.to_csv(rf"{ND}\table_nih_subgroups.csv", index=False)
fig, ax = plt.subplots(figsize=(COL1 * 1.4, 2.5))
ax.bar(range(len(sub)), sub.Mean, color="0.55", edgecolor="k", lw=0.4)
ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub.Group, rotation=35, ha="right")
ax.set(ylabel="Mean AUROC (NIH)", ylim=(0.5, 0.95)); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_nih_subgroups")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

rng = np.random.default_rng(1); NS = 1500
ci_ = cxt.drop_duplicates("patient_id").sample(NS, random_state=1).index.values
ni_ = nih.drop_duplicates("patient_id").sample(NS, random_state=1).index.values
with ThreadPoolExecutor(12) as pool:
    Xc = np.stack(list(pool.map(load224, cxt.img_path.values[ci_])))
Xn = np.stack([arr[i] for i in ni_])
X = np.concatenate([Xc, Xn]); src = np.r_[np.zeros(NS), np.ones(NS)].astype(int)

m0 = load_model(0)
@torch.no_grad()
def feats(a, bs=256):
    out = []
    for i in range(0, len(a), bs):
        with torch.autocast("cuda"):
            out.append(m0.forward_features(prep(a[i:i + bs])).float().mean((2, 3)).cpu())
    return torch.cat(out).numpy()
F = feats(X)

def src_auc(Z_):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    p = cross_val_predict(clf, Z_, src, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    return roc_auc_score(src, p)
Xf = X.astype(np.float32)
stats = np.c_[Xf.mean((1, 2)), Xf.std((1, 2)), np.percentile(Xf.reshape(len(X), -1), 1, axis=1),
              np.percentile(Xf.reshape(len(X), -1), 99, axis=1),
              Xf[:, :12].mean((1, 2)), Xf[:, -12:].mean((1, 2)), Xf[:, :, :12].mean((1, 2)), Xf[:, :, -12:].mean((1, 2))]
pix = Xf.reshape(len(X), 28, 8, 28, 8).mean((2, 4)).reshape(len(X), -1)
src_tab = pd.DataFrame({"Source-classifier AUROC": {"Brightness/contrast/border statistics": src_auc(stats),
                        "Downsampled pixels (28x28)": src_auc(pix), "DenseNet121 features (1024-d)": src_auc(F)}})
display(src_tab.round(3)); src_tab.to_csv(rf"{ND}\table_source_classifier.csv")

fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.3))
for j, (v, nm) in enumerate([(Xf.mean((1, 2)), "Mean intensity"), (Xf.std((1, 2)), "Contrast (std)")]):
    ax[j].hist(v[:NS], 40, alpha=0.6, density=True, label="CheXpert", color="0.3")
    ax[j].hist(v[NS:], 40, alpha=0.6, density=True, label="NIH", color="0.7")
    ax[j].set(xlabel=nm, ylabel="Density"); ax[j].legend(frameon=False)
Zt = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(F))
ax[2].scatter(Zt[:NS, 0], Zt[:NS, 1], s=2, c="0.2", label="CheXpert"); ax[2].scatter(Zt[NS:, 0], Zt[NS:, 1], s=2, c="tab:orange", label="NIH")
ax[2].set(xlabel="t-SNE 1", ylabel="t-SNE 2"); ax[2].legend(frameon=False, markerscale=4)
for a in ax: a.grid(True, alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_appearance_shift")

In [ ]:
LBL = LABELS.index("Pleural Effusion")
thr0 = np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed0_results\thresholds.npy")[LBL]
def gradcam(model, a, cls):
    fm_ = {}
    h = model.features.register_forward_hook(lambda m, i, o: fm_.__setitem__("o", o))
    model.zero_grad(); out = model(prep(a[None])); h.remove()
    fm = fm_["o"]; g = torch.autograd.grad(out[0, cls], fm)[0]
    cam = torch.relu((g.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy()

rng = np.random.default_rng(3); K = 4
cx_tp = np.where((Yc[:, LBL] == 1) & (Pc[0][:, LBL] >= thr0))[0]
nh_tp = np.where((Yn[:, LBL] == 1) & (NP[0][:, LBL] >= thr0))[0]
nh_fn = np.where((Yn[:, LBL] == 1) & (NP[0][:, LBL] < thr0))[0]
rows_ = [("CheXpert, correct", [load224(cxt.img_path[i]) for i in rng.choice(cx_tp, K, replace=False)]),
         ("NIH, correct", [arr[i] for i in rng.choice(nh_tp, K, replace=False)]),
         ("NIH, missed", [arr[i] for i in rng.choice(nh_fn, K, replace=False)])]
fig, ax = plt.subplots(3, K, figsize=(COL2, 5.2))
for r, (nm, imgs) in enumerate(rows_):
    for c, im_ in enumerate(imgs):
        cam = gradcam(m0, np.asarray(im_), LBL)
        ax[r, c].imshow(im_, cmap="gray"); ax[r, c].imshow(cam, cmap="jet", alpha=0.4); ax[r, c].axis("off")
    ax[r, 0].set_title(nm, loc="left", fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_gradcam_effusion")

In [ ]:
tv = ~off
rows = []
for nm, m in [("NIH train_val", tv), ("NIH official test", off), ("NIH all", np.ones(len(nih), bool))]:
    r = {"Subset": nm, "N": int(m.sum()), "AP share": (nih["view"][m] == "AP").mean(), "Mean age": nih.Age[m].mean()}
    for i, l in enumerate(LABELS):
        r["prev " + l] = Yn[m, i].mean()
        r["AUC " + l] = np.mean([fauc(Yn[m, i], NP[s][m, i]) for s in SEEDS])
    r["AUC mean"] = np.mean([r["AUC " + l] for l in LABELS]); rows.append(r)
display(pd.DataFrame(rows).set_index("Subset").T.round(3))
print(res.groupby("seed")["NIH (official test)"].mean().round(4))

In [ ]:
rows = []
for i, l in enumerate(LABELS):
    raw = cxt[l + "_raw"].values
    for nm, mask in [("all test images", np.ones(len(cxt), bool)), ("uncertain removed", raw != 1)]:
        rows.append({"Label": l, "Subset": nm, "N": int(mask.sum()),
                     "AUROC": np.mean([fauc(Yc[mask, i], Pc[s][mask, i]) for s in SEEDS])})
t = pd.DataFrame(rows)
display(t.pivot(index="Label", columns="Subset", values="AUROC").loc[LABELS].round(3))
display(t.pivot(index="Label", columns="Subset", values="N").loc[LABELS])

In [ ]:
from scipy.stats import mannwhitneyu
M = int(0.15 * 224)
edge = np.ones((224, 224), bool); edge[M:-M, M:-M] = False
bottom = np.zeros((224, 224), bool); bottom[149:, :] = True

rng = np.random.default_rng(7); K2 = 200
sets = {"CheXpert, correct": ("cx", rng.choice(cx_tp, K2, replace=False)),
        "NIH, correct": ("nih", rng.choice(nh_tp, K2, replace=False)),
        "NIH, missed": ("nih", rng.choice(nh_fn, K2, replace=False))}
stat, rows = {}, []
for nm, (src_, idx) in sets.items():
    ef, bf, zero = [], [], 0
    for i in tqdm(idx, desc=nm, leave=False):
        im_ = load224(cxt.img_path[i]) if src_ == "cx" else arr[i]
        cam = gradcam(m0, np.asarray(im_), LBL); s = cam.sum()
        if s <= 0:
            zero += 1; continue                       # no positive evidence for the class
        ef.append(cam[edge].sum() / s); bf.append(cam[bottom].sum() / s)
    ef, bf = np.array(ef), np.array(bf); stat[nm] = (ef, bf)
    rows.append({"Group": nm, "Empty heatmaps": f"{zero}/{K2}",
                 "Edge fraction": f"{ef.mean():.3f} ± {ef.std(ddof=1)/np.sqrt(len(ef)):.3f}" if len(ef) > 1 else "n/a",
                 "Bottom-third fraction": f"{bf.mean():.3f} ± {bf.std(ddof=1)/np.sqrt(len(bf)):.3f}" if len(bf) > 1 else "n/a"})
tab_cam = pd.DataFrame(rows).set_index("Group"); display(tab_cam)
print("Uniform-heat baseline: edge 0.50, bottom third 0.33")
a_, b_ = stat["NIH, correct"], stat["NIH, missed"]
if len(b_[0]) > 1:
    for j, nm in enumerate(["Edge fraction", "Bottom-third fraction"]):
        print(f"{nm}: NIH correct vs missed, Mann-Whitney p = {mannwhitneyu(a_[j], b_[j]).pvalue:.2g}")
tab_cam.to_csv(rf"{ND}\table_gradcam_regions.csv")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
for j, (nm, base) in enumerate([("Edge fraction", 0.50), ("Bottom-third fraction", 0.33)]):
    ax[j].boxplot([stat[k][j] for k in stat if len(stat[k][j])], labels=[k.replace(", ", "\n") for k in stat if len(stat[k][j])], showfliers=False)
    ax[j].axhline(base, color="k", ls=":", lw=0.6); ax[j].set(ylabel=nm); ax[j].grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_gradcam_regions")

In [ ]:
XC = np.stack(list(ThreadPoolExecutor(12).map(load224, cxt.img_path.values)))     # all CheXpert test, ~1 GB
rng = np.random.default_rng(11); sub = np.sort(rng.choice(len(nih), 30000, replace=False))
XN, YN = np.asarray(arr[sub]), Yn[sub]

@torch.no_grad()
def masked_pred(model, a, frac, bs=256):
    m = int(224 * frac); out = []
    for i in range(0, len(a), bs):
        x = torch.from_numpy(np.array(a[i:i + bs])).cuda().float()
        if m > 0:
            fill = x.flatten(1).median(1).values.view(-1, 1, 1)
            x[:, :m, :] = fill; x[:, -m:, :] = fill; x[:, :, :m] = fill; x[:, :, -m:] = fill
        x = (x.div(255).unsqueeze(1).repeat(1, 3, 1, 1) - MEAN) / STD
        with torch.autocast("cuda"):
            o = model(x)
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

rows = []
for fr in [0, 0.05, 0.10, 0.15]:
    Pc_, Pn_ = masked_pred(m0, XC, fr), masked_pred(m0, XN, fr)
    r = {"Margin masked": f"{int(fr*100)}%"}
    for i, l in enumerate(LABELS):
        r["CX " + l] = fauc(Yc[:, i], Pc_[:, i]); r["NIH " + l] = fauc(YN[:, i], Pn_[:, i])
    r["CheXpert mean"] = np.mean([r["CX " + l] for l in LABELS]); r["NIH mean"] = np.mean([r["NIH " + l] for l in LABELS])
    rows.append(r)
mk_ = pd.DataFrame(rows).set_index("Margin masked")
mk_["CheXpert change"] = mk_["CheXpert mean"] - mk_["CheXpert mean"].iloc[0]
mk_["NIH change"] = mk_["NIH mean"] - mk_["NIH mean"].iloc[0]
display(mk_[["CheXpert mean", "NIH mean", "CheXpert change", "NIH change"]].round(4))
display(mk_[[c for c in mk_.columns if c.startswith(("CX ", "NIH ")) and "mean" not in c]].T.round(3))
mk_.to_csv(rf"{ND}\table_border_masking.csv")

fig, ax = plt.subplots(figsize=(COL1 * 1.2, 2.5))
ax.plot(range(len(mk_)), mk_["CheXpert mean"], "-o", ms=3, label="CheXpert")
ax.plot(range(len(mk_)), mk_["NIH mean"], "--s", ms=3, label="NIH (30k subset)")
ax.set_xticks(range(len(mk_))); ax.set_xticklabels(mk_.index)
ax.set(xlabel="Border masked", ylabel="Mean AUROC"); ax.legend(frameon=False); ax.grid(True, alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_border_masking")